In [1]:
# Run this cell once if the libraries are not installed
!pip install requests pandas beautifulsoup4 lxml

import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

In [2]:
locations = {
    "Bangalore Central": (12.9716, 77.5946),
    "Whitefield": (12.9698, 77.7500),
    "Electronic City": (12.8452, 77.6602),
    "Yelahanka": (13.1007, 77.5963),
    "Rajajinagar": (12.9911, 77.5553)
}

print("Locations:")
for name, (lat, lon) in locations.items():
    print(f"{name}: Latitude={lat}, Longitude={lon}")

Locations:
Bangalore Central: Latitude=12.9716, Longitude=77.5946
Whitefield: Latitude=12.9698, Longitude=77.75
Electronic City: Latitude=12.8452, Longitude=77.6602
Yelahanka: Latitude=13.1007, Longitude=77.5963
Rajajinagar: Latitude=12.9911, Longitude=77.5553


In [3]:
weather_data = []

for location_name, (latitude, longitude) in locations.items():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": (pd.Timestamp.today() - pd.Timedelta(days=21)).strftime("%Y-%m-%d"),
        "end_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
        "hourly": "temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Kolkata"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()

        df = pd.DataFrame({
            "time": data["hourly"]["time"],
            "temperature": data["hourly"]["temperature_2m"],
            "humidity": data["hourly"]["relative_humidity_2m"]
        })

        df["location"] = location_name
        df["latitude"] = latitude
        df["longitude"] = longitude

        weather_data.append(df)

        print(f"{location_name}: Weather data collected")

    else:
        print(f"{location_name}: Error {response.status_code}")

    time.sleep(1)

weather_df = pd.concat(weather_data, ignore_index=True)

weather_df.head()

Bangalore Central: Weather data collected
Whitefield: Weather data collected
Electronic City: Weather data collected
Yelahanka: Weather data collected
Rajajinagar: Weather data collected


,time,temperature,humidity,location,latitude,longitude
0,2026-07-29T00:00,21.9,87,Bangalore Central,12.9716,77.5946
1,2026-07-29T01:00,21.7,88,Bangalore Central,12.9716,77.5946
2,2026-07-29T02:00,21.7,88,Bangalore Central,12.9716,77.5946
3,2026-07-29T03:00,21.6,88,Bangalore Central,12.9716,77.5946
4,2026-07-29T04:00,21.7,87,Bangalore Central,12.9716,77.5946


In [4]:
weather_df.to_csv("bengaluru_weather.csv", index=False)

print("Weather data saved successfully!")
print("File: bengaluru_weather.csv")
print("Rows:", len(weather_df))

Weather data saved successfully!
File: bengaluru_weather.csv
Rows: 2640


In [5]:
print(weather_df.info())
print("\nWeather Summary:")
display(weather_df.describe())

print("\nData by location:")
display(weather_df.groupby("location")[["temperature", "humidity"]].mean())

<class 'pandas.DataFrame'>
RangeIndex: 2640 entries, 0 to 2639
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   time         2640 non-null   str    
 1   temperature  2640 non-null   float64
 2   humidity     2640 non-null   int64  
 3   location     2640 non-null   str    
 4   latitude     2640 non-null   float64
 5   longitude    2640 non-null   float64
dtypes: float64(3), int64(1), str(2)
memory usage: 123.9 KB
None

Weather Summary:


,temperature,humidity,latitude,longitude
count,2640.000000,2640.000000,2640.000000,2640.000000
mean,23.812045,77.382576,12.975680,77.631280
std,2.754557,14.398218,0.081187,0.068237
min,19.600000,41.000000,12.845200,77.555300
25%,21.500000,65.000000,12.969800,77.594600
50%,23.100000,81.000000,12.971600,77.596300
75%,26.000000,90.000000,12.991100,77.660200
max,31.300000,100.000000,13.100700,77.750000



Data by location:


,temperature,humidity
location,,
Bangalore Central,23.675758,77.825758
Electronic City,23.602462,78.903409
Rajajinagar,23.675758,77.825758
Whitefield,24.321591,74.539773
Yelahanka,23.784659,77.818182


In [6]:
air_quality_data = []

for location_name, (latitude, longitude) in locations.items():

    url = "https://air-quality-api.open-meteo.com/v1/air-quality"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": (pd.Timestamp.today() - pd.Timedelta(days=30)).strftime("%Y-%m-%d"),
        "end_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
        "hourly": "pm10,pm2_5,carbon_monoxide",
        "timezone": "Asia/Kolkata"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()

        df = pd.DataFrame({
            "time": data["hourly"]["time"],
            "PM10": data["hourly"]["pm10"],
            "PM2.5": data["hourly"]["pm2_5"],
            "CO": data["hourly"]["carbon_monoxide"]
        })

        df["location"] = location_name
        df["latitude"] = latitude
        df["longitude"] = longitude

        air_quality_data.append(df)

        print(f"{location_name}: Air quality data collected")

    else:
        print(f"{location_name}: Error {response.status_code}")

    time.sleep(1)

air_quality_df = pd.concat(air_quality_data, ignore_index=True)

air_quality_df.head()

Bangalore Central: Air quality data collected
Whitefield: Air quality data collected
Electronic City: Air quality data collected
Yelahanka: Air quality data collected
Rajajinagar: Air quality data collected


,time,PM10,PM2.5,CO,location,latitude,longitude
0,2026-07-20T00:00,10.5,7.6,210.0,Bangalore Central,12.9716,77.5946
1,2026-07-20T01:00,7.4,5.6,174.0,Bangalore Central,12.9716,77.5946
2,2026-07-20T02:00,6.0,4.5,151.0,Bangalore Central,12.9716,77.5946
3,2026-07-20T03:00,5.4,4.0,137.0,Bangalore Central,12.9716,77.5946
4,2026-07-20T04:00,5.0,3.7,136.0,Bangalore Central,12.9716,77.5946


In [7]:
air_quality_df.to_csv("bengaluru_air_quality.csv", index=False)

print("Air quality data saved successfully!")
print("File: bengaluru_air_quality.csv")
print("Rows:", len(air_quality_df))

Air quality data saved successfully!
File: bengaluru_air_quality.csv
Rows: 3720


In [8]:
print(air_quality_df.info())

print("\nAir Quality Summary:")
display(air_quality_df.describe())

print("\nAverage pollutant levels by location:")
display(
    air_quality_df.groupby("location")[["PM10", "PM2.5", "CO"]].mean()
)

<class 'pandas.DataFrame'>
RangeIndex: 3720 entries, 0 to 3719
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   time       3720 non-null   str    
 1   PM10       3720 non-null   float64
 2   PM2.5      3720 non-null   float64
 3   CO         3720 non-null   float64
 4   location   3720 non-null   str    
 5   latitude   3720 non-null   float64
 6   longitude  3720 non-null   float64
dtypes: float64(5), str(2)
memory usage: 203.6 KB
None

Air Quality Summary:


,PM10,PM2.5,CO,latitude,longitude
count,3720.000000,3720.000000,3720.000000,3720.000000,3720.000000
mean,11.556048,6.963737,218.396237,12.975680,77.631280
std,6.608158,3.352017,96.996090,0.081182,0.068233
min,1.800000,1.600000,103.000000,12.845200,77.555300
25%,6.700000,4.400000,147.000000,12.969800,77.594600
50%,10.600000,6.400000,188.000000,12.971600,77.596300
75%,15.000000,8.800000,261.000000,12.991100,77.660200
max,83.800000,29.900000,690.000000,13.100700,77.750000



Average pollutant levels by location:


,PM10,PM2.5,CO
location,,,
Bangalore Central,11.445027,6.995161,276.864247
Electronic City,11.445027,6.995161,146.723118
Rajajinagar,11.445027,6.995161,276.864247
Whitefield,11.445027,6.995161,212.063172
Yelahanka,12.000134,6.838038,179.466398


In [9]:
books = []

base_url = "https://books.toscrape.com/catalogue/page-1.html"
current_url = base_url

while current_url:

    response = requests.get(current_url)

    if response.status_code != 200:
        print("Error:", response.status_code)
        break

    soup = BeautifulSoup(response.text, "html.parser")

    # Find all books on the current page
    book_items = soup.select("article.product_pod")

    for book in book_items:

        # Book title
        title = book.h3.a["title"]

        # Price
        price = book.select_one(".price_color").text.strip()

        # Rating
        rating_class = book.select_one("p.star-rating")["class"]
        rating = rating_class[1]

        books.append({
            "Title": title,
            "Price": price,
            "Rating": rating
        })

    print(f"Scraped {len(book_items)} books from:", current_url)

    # Find next page
    next_button = soup.select_one("li.next a")

    if next_button:
        current_url = urljoin(current_url, next_button["href"])
    else:
        current_url = None

    time.sleep(0.5)

books_df = pd.DataFrame(books)

books_df.head()

Scraped 20 books from: https://books.toscrape.com/catalogue/page-1.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-2.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-3.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-4.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-5.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-6.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-7.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-8.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-9.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-10.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-11.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-12.html
Scraped 20 books from: https://books.toscrape.com/catalogue/page-13.html
Scraped 20 books from: https://books.toscrape.com/catalogue/

,Title,Price,Rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five


In [11]:
books_df.to_csv("books_data.csv", index=False)

print("Bookstore data saved successfully!")
print("File: books_data.csv")
print("Total books:", len(books_df))

Bookstore data saved successfully!
File: books_data.csv
Total books: 1000


In [12]:
print("First 10 books:")
display(books_df.head(10))

print("\nRating distribution:")
display(books_df["Rating"].value_counts())

print("\nTotal books scraped:", len(books_df))

First 10 books:


,Title,Price,Rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five
5,The Requiem Red,Â£22.65,One
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four
9,The Black Maria,Â£52.15,One



Rating distribution:


Rating
One      226
Three    203
Five     196
Two      196
Four     179
Name: count, dtype: int64


Total books scraped: 1000


In [13]:
import os

files = [
    "bengaluru_weather.csv",
    "bengaluru_air_quality.csv",
    "books_data.csv"
]

print("Generated files:\n")

for file in files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024
        print(f"✓ {file}  ({size:.2f} KB)")
    else:
        print(f"✗ {file} not found")

Generated files:

✓ bengaluru_weather.csv  (141.85 KB)
✓ bengaluru_air_quality.csv  (224.30 KB)
✓ books_data.csv  (54.92 KB)
